# 02 - HMM Regime Training

This notebook mirrors `backend/hmm_model.py` logic for demo-friendly experimentation.

In [ ]:
import duckdb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from hmmlearn.hmm import GaussianHMM

conn = duckdb.connect("../data/finvizaard.duckdb")

In [ ]:
ticker = "AAPL"
df = conn.execute("""
SELECT ts, close
FROM prices
WHERE ticker = ?
ORDER BY ts
""", [ticker]).df()

df["ret_1"] = df["close"].pct_change().fillna(0.0)
df["vol_10"] = df["ret_1"].rolling(10).std().fillna(0.0)
X = np.column_stack([df["ret_1"].to_numpy(), df["vol_10"].to_numpy()])
df.tail()

In [ ]:
model = GaussianHMM(
    n_components=3,
    covariance_type="diag",
    n_iter=200,
    random_state=42,
)
model.fit(X)
states = model.predict(X)
df["state"] = states

state_means = df.groupby("state")["ret_1"].mean().sort_values()
labels = ["bear", "sideways", "bull"]
state_to_label = {int(s): labels[i] for i, s in enumerate(state_means.index.tolist())}
df["regime"] = df["state"].map(state_to_label)

state_means, state_to_label

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
for regime, group in df.groupby("regime"):
    ax.scatter(group["ts"], group["close"], s=6, label=regime, alpha=0.7)
ax.set_title(f"{ticker} Close by Detected Regime")
ax.legend()
plt.tight_layout()

## Demo Notes
- Explain that HMM uses hidden states inferred from return + volatility patterns.
- Show mapping from state index to semantic labels (`bear/sideways/bull`).